In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------- paths --------
SIM = Path("../../../data/simulation/")

INPUTS = [
    ("engine_normal_load_X.npy",   "engine_normal_load_y.npy"),
    ("engine_high_load_X.npy",     "engine_high_load_y.npy"),
    ("engine_critical_load_X.npy", "engine_critical_load_y.npy"),
    ("engine_off_X.npy",           "engine_off_y.npy"),
    ("engine_start_X.npy",         "engine_start_y.npy"),
]

OUT_X   = SIM / "engine_occ_train_X.npy"
OUT_Y   = SIM / "engine_occ_train_y.npy"
OUT_CSV = SIM / "engine_occ_train.csv"

FEATURES = ["Temperature","Pressure","RPM","Vibration"]
rng = np.random.default_rng(42)

# -------- OCC errors (only out_of_range now) --------
def _oor(name, size):
    """Generate uncalibrated out-of-range values for a feature."""
    if name == "Temperature":
        return (rng.uniform(-90, -30, size) if rng.random() < 0.5 else rng.uniform(165, 300, size))
    if name == "Pressure":
        return (rng.uniform(-6, 0.01, size) if rng.random() < 0.5 else rng.uniform(1.5, 6, size))
    if name == "RPM":
        return (rng.uniform(-2000, -0.1, size) if rng.random() < 0.5 else rng.uniform(12000, 100000, size))
    return (rng.uniform(-20, -0.1, size) if rng.random() < 0.5 else rng.uniform(2, 20, size))

def err_out_of_range(col, feat):
    """Corrupt the entire column with out-of-range (uncalibrated) values."""
    return _oor(feat, col.shape[0]).astype(col.dtype)

# Keep structure for extensibility
ERROR_FUNCS = {"out_of_range": err_out_of_range}
ERROR_NAMES = ["out_of_range"]
ERROR_PROBS = np.array([1.0], dtype=float)

def apply_occ(win_T4):
    """Apply OCC corruption to a (T,4) window."""
    w = win_T4.astype(np.float32).copy()
    k = int(rng.integers(1, 5))  # number of variables to corrupt (1..4)
    var_idx = rng.choice(4, size=k, replace=False)
    for vi in var_idx:
        feat = FEATURES[vi]
        chosen = rng.choice(ERROR_NAMES, p=ERROR_PROBS)  # always 'out_of_range'
        w[:, vi] = ERROR_FUNCS[chosen](w[:, vi], feat)
    return w

# -------- build datasets --------
X_total = []
y_total = []

for x_name, y_name in INPUTS:
    X = np.load(SIM / x_name, allow_pickle=True)
    _ = np.load(SIM / y_name, allow_pickle=True)  # just to ensure file exists

    assert X.ndim == 3 and X.shape[2] == 4, f"Bad X shape {X.shape} @ {x_name}"

    for i in range(X.shape[0]):
        w = X[i]             # (T,4)
        w_occ = apply_occ(w) # OCC applied
        T = w_occ.shape[0]

        X_total.append(w_occ)

        # Label handling (all OCC samples are 'Unknown')
        if "engine_off" in x_name.lower():
            y_total.append(np.repeat("Unknown", 30))  # fixed length for off-state
        else:
            y_total.append(np.repeat("Unknown", T))   # match sequence length

# -------- save ragged arrays --------
X_total = np.array(X_total, dtype=object)
y_total = np.array(y_total, dtype=object)

np.save(OUT_X, X_total, allow_pickle=True)
np.save(OUT_Y, y_total, allow_pickle=True)

# -------- CSV export --------
rows = []
for i, (X_seq, y_seq) in enumerate(zip(X_total, y_total)):
    for t in range(X_seq.shape[0]):
        rows.append({
            "Sequence":    i + 1,
            "Time":        t + 1,
            "Temperature": float(X_seq[t, 0]),
            "Pressure":    float(X_seq[t, 1]),
            "RPM":         float(X_seq[t, 2]),
            "Vibration":   float(X_seq[t, 3]),
            "State":       y_seq[t],
        })

pd.DataFrame(rows, columns=["Sequence","Time","Temperature","Pressure","RPM","Vibration","State"]).to_csv(OUT_CSV, index=False)

print("✅ Engine OCC dataset created (only 'out_of_range', no sentinel error_value).")


✅ Engine OCC dataset created (only 'out_of_range', no sentinel error_value).
